## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import config

## Functions

In [2]:
def load_yahoo_data(ticker, input_dir):
    """
    Load data (date, prices, dividend and volume) from cleaned Yahoo Finance CSV file.
    """

    input_file = input_dir / f"{ticker}.csv"

    if not input_file.exists():
        raise FileNotFoundError(f"Yahoo Finance file not found for {ticker}: {input_file}")

    data = pd.read_csv(input_file)
    data["Date"] = pd.to_datetime(data["Date"])
    data = data.sort_values("Date").reset_index(drop=True)
    data["Ticker"] = ticker

    # Keep Monday-Friday observations only (remove weekend days for crypto assets)
    data = data[data["Date"].dt.dayofweek < 5].copy()

    return data

def construct_return_momentum_features(data):
    """
    Construct daily returns and momentum features for one asset.
    """

    data = data.copy()
    price = data["Adj Close"]

    # Daily return
    data["Return_1d"] = price.pct_change()

    # Momentum returns
    for window in config.MOMENTUM_WINDOWS:
        data[f"Momentum_{window}d"] = (price / price.shift(window) - 1)

    return data

def construct_trend_features(data):
    """
    Construct moving-average based trend features.
    """

    data = data.copy()
    price = data["Adj Close"]

    # Moving averages
    for window in config.TREND_WINDOWS:
        data[f"MA_{window}d"] = (price.rolling(window=window).mean())

        # Relative distance from moving average
        data[f"Price_to_MA_{window}d"] = (price / data[f"MA_{window}d"] - 1)

    # Moving-average crossover signals
    data["MA_20_50_Spread"] = (data["MA_20d"] / data["MA_50d"] - 1)
    data["MA_50_200_Spread"] = (data["MA_50d"] / data["MA_200d"] - 1)

    return data

def construct_mean_reversion_features(data):
    """
    Construct rolling mean-reversion and z-score features.
    """

    data = data.copy()
    price = data["Adj Close"]

    for window in config.MEAN_REVERSION_WINDOWS:

        rolling_mean = price.rolling(window=window).mean()
        rolling_std = price.rolling(window=window).std()

        # Relative distance from rolling mean
        data[f"Mean_Reversion_{window}d"] = (price / rolling_mean - 1)

        # Price z-score
        data[f"Price_ZScore_{window}d"] = ((price - rolling_mean) / rolling_std)

    return data

def construct_volatility_features(data):
    """
    Construct rolling annualized realized-volatility features.
    """

    data = data.copy()
    returns = data["Return_1d"]

    for window in config.VOLATILITY_WINDOWS:

        data[f"Volatility_{window}d"] = (returns.rolling(window=window).std()* np.sqrt(252))

    return data

def construct_drawdown_features(data):
    """
    Construct rolling drawdown features.
    """

    data = data.copy()
    price = data["Adj Close"]

    for window in config.DRAWDOWN_WINDOWS:

        rolling_high = price.rolling(window=window).max()
        data[f"Drawdown_{window}d"] = (price / rolling_high - 1)

    return data

def construct_volume_features(data):
    """
    Construct normalized volume features.
    """

    data = data.copy()

    volume = data["Volume"]

    for window in config.VOLUME_WINDOWS:

        volume_mean = volume.rolling(window=window).mean()
        volume_std = volume.rolling(window=window).std()

        # Current volume relative to average volume
        data[f"Volume_Ratio_{window}d"] = (volume / volume_mean)

        # Volume z-score
        data[f"Volume_ZScore_{window}d"] = ((volume - volume_mean) / volume_std)

    return data

def construct_target(data):
    """
    Construct the forward-return classification target.

    Target:
        1 = positive forward return
        0 = zero or negative forward return
    """

    data = data.copy()
    price = data["Adj Close"]

    # Forward return
    data[f"Forward_Return_{config.FORWARD_HORIZON}d"] = (price.shift(-config.FORWARD_HORIZON) / price - 1)

    # Classification target
    data[f"Target_{config.FORWARD_HORIZON}d"] = (data[f"Forward_Return_{config.FORWARD_HORIZON}d"] > 0).astype("Int64")

    return data

In [3]:
def load_fred_data(series_id, input_dir):
    """
    Load data (date, value) from a cleaned FRED CSV file.
    """

    input_file = input_dir / f"{series_id}.csv"

    if not input_file.exists():
        raise FileNotFoundError(f"FRED file not found for {series_id}: {input_file}")

    data = pd.read_csv(input_file)
    data["Date"] = pd.to_datetime(data["Date"])
    data = (data.sort_values("Date").reset_index(drop=True))
    data["Series"] = series_id

    # Keep Monday-Friday observations only
    data = data[data["Date"].dt.dayofweek < 5].copy()

    return data

def load_yahoo_adj_close_to_calendar(ticker, calendar, input_dir):
    """
    Load Yahoo Finance adjusted close prices and align them
    to the specified trading calendar.
    """

    data = load_yahoo_data(
        ticker=ticker,
        input_dir=input_dir
    )

    adj_close = (
        data[["Date", "Adj Close"]]
        .drop_duplicates(subset=["Date"], keep="last")
        .set_index("Date")
        .sort_index()["Adj Close"]
    )

    # Align prices to the trading calendar
    adj_close = adj_close.reindex(calendar)

    # Forward-fill occasional missing observations
    adj_close = adj_close.ffill()
    adj_close.name = ticker

    return adj_close

def align_fred_series_to_calendar(data, calendar, value_column, lag):
    """
    Align a FRED series to a trading-day calendar. Missing values on trading
    days are forward-filled using the most recent FRED observation. The final
    series is then shifted by a lag measured in trading-day units.
    """

    series_data = (
        data[["Date", value_column]]
        .dropna(subset=[value_column])
        .drop_duplicates(subset=["Date"], keep="last")
        .sort_values("Date")
        .set_index("Date")
    )

    # Include both FRED observation dates and trading dates
    combined_index = (
        series_data.index
        .union(calendar)
        .sort_values()
    )

    # Forward-fill the latest available FRED observation
    aligned_data = (
        series_data
        .reindex(combined_index)
        .ffill()
        .reindex(calendar)
    )

    # Apply lag using the trading calendar
    lagged_series = aligned_data.shift(lag)

    return lagged_series

def construct_fred_feature_dataframe(series_ids, calendar, input_dir, lag):
    """
    Load and align multiple FRED series to a trading calendar.
    """

    feature_data = pd.DataFrame(index=calendar)

    for series_id in series_ids:

        print(f"Processing {series_id} FRED feature...")

        # Load FRED data
        fred_data = load_fred_data(
            series_id=series_id,
            input_dir=input_dir
        )

        # Align FRED data to the trading calendar
        feature_data[series_id] = (
            align_fred_series_to_calendar(
                data=fred_data,
                calendar=calendar,
                value_column=series_id,
                lag=lag
            )
        )

    return feature_data

def construct_macro_rolling_features(macro_features, monthly_columns, rolling_windows, change_windows):
    """
    Construct rolling features for daily macroeconomic variables.
    Features: rolling mean, rolling std dev, rolling z-score, changes
    """

    macro_features = macro_features.copy()

    if monthly_columns is None:
        monthly_columns = []

    # Preserve the original columns before adding new features
    base_macro_columns = macro_features.columns.tolist()
    new_feature_frames = []

    for column in base_macro_columns:

        # Leave monthly macro features unchanged
        if column in monthly_columns:
            continue

        series = macro_features[column]
        feature_dict = {}

        # Rolling features for daily macro variables
        for window in rolling_windows:

            # Rolling mean
            rolling_mean = series.rolling(window=window, min_periods=window).mean()
            feature_dict[f"{column}_RollingMean_{window}d"] = rolling_mean

            # Rolling standard deviation
            rolling_std = series.rolling(window=window, min_periods=window).std()
            feature_dict[f"{column}_RollingStd_{window}d"] = rolling_std

            # Rolling z-score
            feature_dict[f"{column}_RollingZScore_{window}d"] = (series - rolling_mean).div(
                rolling_std.replace(0, np.nan))

        # Change features
        for window in change_windows:
            feature_dict[f"{column}_Change_{window}d"] = series.diff(window)

        # Convert dict to DataFrame
        new_feature_frames.append(pd.DataFrame(feature_dict))

    # Concatenate all new features to avoid fragmentation
    if new_feature_frames:
        new_features = pd.concat(new_feature_frames, axis=1)
        macro_features = pd.concat([macro_features, new_features], axis=1)

    return macro_features

## Create Asset Features

In [4]:
# Define input data directories
YAHOO_INPUT_DIR = config.INPUT_DATA_DIR / "yahoo"
FRED_INPUT_DIR = config.INPUT_DATA_DIR / "fred"
FAMA_FRENCH_INPUT_DIR = config.INPUT_DATA_DIR / "fama_french"

# Research data output directory
FEATURES_DATA_DIR = config.INPUT_DATA_DIR.parent / "features_data"
FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Define output file
ASSET_FEATURES_FILE = FEATURES_DATA_DIR / "asset_features.csv"

# Load the master investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

# Get Yahoo assets
yahoo_universe = universe[universe["Source"].str.strip().str.lower() == "yahoo finance"].copy()
tickers = yahoo_universe["Ticker"].tolist()

# Construct features for all Yahoo assets
ticker_feature_data = []

for ticker in tickers:

    print(f"Constructing features for {ticker}...")

    data = load_yahoo_data(ticker=ticker, input_dir=YAHOO_INPUT_DIR)
    data = construct_return_momentum_features(data)
    data = construct_trend_features(data)
    data = construct_mean_reversion_features(data)
    data = construct_volatility_features(data)
    data = construct_drawdown_features(data)
    data = construct_volume_features(data)
    data = construct_target(data)

    ticker_feature_data.append(data)

# Combine assets and apply start date
asset_features = pd.concat(ticker_feature_data, ignore_index=True)
asset_features = asset_features.sort_values(["Date", "Ticker"]).reset_index(drop=True)
asset_features = asset_features[asset_features["Date"] >= config.START_DATE].copy()

# Keep features and target only
asset_features = (asset_features.drop(
    columns = ["Adj Close", "Close", "Capital Gains", "Dividends", "High", "Low", "Open", "Stock Splits", "Volume"]
).copy())

# Remove rows with missing values
asset_features = asset_features.dropna(subset = ["Momentum_252d", f"Forward_Return_{config.FORWARD_HORIZON}d"]).copy()

# Reset index after filtering
asset_features = asset_features.reset_index(drop=True)

# Save the file
asset_features.to_csv(ASSET_FEATURES_FILE, index=False)
print(f"Saved asset features to: {ASSET_FEATURES_FILE}")

Constructing features for SPY...
Constructing features for VT...
Constructing features for QQQ...
Constructing features for VTV...
Constructing features for VUG...
Constructing features for IWM...
Constructing features for DVY...
Constructing features for VEA...
Constructing features for VGK...
Constructing features for FEZ...
Constructing features for EWJ...
Constructing features for FXI...
Constructing features for AFK...
Constructing features for VWO...
Constructing features for IPO...
Constructing features for SHY...
Constructing features for IEF...
Constructing features for TLT...
Constructing features for BND...
Constructing features for TIP...
Constructing features for VCSH...
Constructing features for LQD...
Constructing features for HYG...
Constructing features for EMB...
Constructing features for BWX...
Constructing features for VNQ...
Constructing features for RWX...
Constructing features for IGF...
Constructing features for GLD...
Constructing features for SLV...
Constructi

## Create Macro Features

In [5]:
# Load series configurations
YAHOO_SERIES = config.YAHOO_SERIES
FRED_SERIES = config.FRED_SERIES

# Define output file
MACRO_FEATURES_FILE = FEATURES_DATA_DIR / "macro_features.csv"

# Extract series IDs
YAHOO_FEATURES = list(YAHOO_SERIES.keys())
MONTHLY_FEATURES = list(FRED_SERIES["MONTHLY_FEATURES"].keys())
INTEREST_RATE_FEATURES = list(FRED_SERIES["INTEREST_RATE_FEATURES"].keys())
YIELD_CURVE_FEATURES = list(FRED_SERIES["YIELD_CURVE_FEATURES"].keys())
TERM_PREMIUM_FEATURES = list(FRED_SERIES["TERM_PREMIUM_FEATURES"].keys())
INFLATION_FEATURES = list(FRED_SERIES["INFLATION_FEATURES"].keys())
VOLATILITY_FEATURES = list(FRED_SERIES["VOLATILITY_FEATURES"].keys())
CORP_CREDIT_FEATURES = list(FRED_SERIES["CORP_CREDIT_FEATURES"].keys())

# Create trading calendar from SPY data
spy_data = load_yahoo_data(ticker="SPY", input_dir=YAHOO_INPUT_DIR)
spy_calendar = pd.DatetimeIndex(spy_data["Date"].drop_duplicates().sort_values())
spy_calendar = spy_calendar[spy_calendar >= (pd.Timestamp(config.START_DATE) - pd.DateOffset(years=2))]

# Store monthly macro features
monthly_fred_features = construct_fred_feature_dataframe(
    series_ids=MONTHLY_FEATURES,
    calendar=spy_calendar,
    input_dir=FRED_INPUT_DIR,
    lag=config.MONTHLY_FEATURE_LAG
)

# Store interest rate features
interest_rate_fred_features = construct_fred_feature_dataframe(
    series_ids=INTEREST_RATE_FEATURES,
    calendar=spy_calendar,
    input_dir=FRED_INPUT_DIR,
    lag=config.INTEREST_RATE_LAG
)

# Store term premium features
term_premium_fred_features = construct_fred_feature_dataframe(
    series_ids=TERM_PREMIUM_FEATURES,
    calendar=spy_calendar,
    input_dir=FRED_INPUT_DIR,
    lag=config.TERM_PREMIUM_LAG
)

# Store corporate credit features
corp_credit_fred_features = construct_fred_feature_dataframe(
    series_ids=CORP_CREDIT_FEATURES,
    calendar=spy_calendar,
    input_dir=FRED_INPUT_DIR,
    lag=config.CORP_CREDIT_LAG
)

# Store inflation expectation features
inflation_fred_features = construct_fred_feature_dataframe(
    series_ids=INFLATION_FEATURES,
    calendar=spy_calendar,
    input_dir=FRED_INPUT_DIR,
    lag=0
)

# Store volatility features
volatility_fred_features = construct_fred_feature_dataframe(
    series_ids=VOLATILITY_FEATURES,
    calendar=spy_calendar,
    input_dir=FRED_INPUT_DIR,
    lag=0
)

# Store yield curve features
yield_curve_fred_features = construct_fred_feature_dataframe(
    series_ids=YIELD_CURVE_FEATURES,
    calendar=spy_calendar,
    input_dir=FRED_INPUT_DIR,
    lag=0
)

# Combine all FRED feature categories
fred_features = pd.concat(
    [
        monthly_fred_features,
        interest_rate_fred_features,
        yield_curve_fred_features,
        term_premium_fred_features,
        inflation_fred_features,
        volatility_fred_features,
        corp_credit_fred_features,
    ],
    axis=1
)

# Ensure chronological order
fred_features = fred_features.sort_index()

# Add Yahoo features to FRED data
for ticker in YAHOO_FEATURES:

    print(f"Processing {ticker} Yahoo feature...")

    fred_features[ticker] = load_yahoo_adj_close_to_calendar(
        ticker=ticker,
        calendar=fred_features.index,
        input_dir=YAHOO_INPUT_DIR
    )

# Ensure chronological order
macro_features = fred_features.sort_index()

# Construct rolling features for daily macro variables only
macro_features = construct_macro_rolling_features(
    macro_features=macro_features,
    monthly_columns = [column for column in MONTHLY_FEATURES if column in macro_features.columns] + (["DFF"] if "DFF" in macro_features.columns else []),
    rolling_windows=config.MACRO_WINDOWS,
    change_windows=config.CHANGE_WINDOWS
)

# Keep only observations from the start date
macro_features = macro_features.loc[macro_features.index >= pd.Timestamp(config.START_DATE)]

# Count missing values per column and remove them if any
missing_counts = macro_features.isna().sum()
missing_counts = missing_counts[missing_counts > 0]

if not missing_counts.empty:
    macro_features = macro_features.dropna()
    print(f"\nMissing values found and removed from macro features:")
    print(f"{missing_counts}")

# Save the file
macro_features_to_save = (macro_features.reset_index().rename(columns={"index": "Date"}))
macro_features_to_save.to_csv(MACRO_FEATURES_FILE,index=False)
print(f"Saved macro features to: {MACRO_FEATURES_FILE}")

Processing CPIAUCSL FRED feature...
Processing UNRATE FRED feature...
Processing INDPRO FRED feature...
Processing MTSDS133FMS FRED feature...
Processing MTSO133FMS FRED feature...
Processing MVGFD027MNFRBDAL FRED feature...
Processing DFF FRED feature...
Processing DGS3MO FRED feature...
Processing DGS2 FRED feature...
Processing DGS10 FRED feature...
Processing DGS30 FRED feature...
Processing THREEFYTP2 FRED feature...
Processing THREEFYTP5 FRED feature...
Processing THREEFYTP10 FRED feature...
Processing BAA10Y FRED feature...
Processing AAA10Y FRED feature...
Processing T5YIFR FRED feature...
Processing T5YIE FRED feature...
Processing T10YIE FRED feature...
Processing VIXCLS FRED feature...
Processing VXDCLS FRED feature...
Processing T10Y2Y FRED feature...
Processing T10Y3M FRED feature...
Processing DX-Y.NYB Yahoo feature...
Processing CL=F Yahoo feature...
Saved macro features to: features_data\macro_features.csv


In [6]:
# Load asset and macro features
asset_features = pd.read_csv(ASSET_FEATURES_FILE)
macro_features = pd.read_csv(MACRO_FEATURES_FILE)

# Convert Date columns
asset_features["Date"] = pd.to_datetime(asset_features["Date"])
macro_features["Date"] = pd.to_datetime(macro_features["Date"])

# Sort by date
asset_features = asset_features.sort_values(["Date", "Ticker"]).reset_index(drop=True)
macro_features = macro_features.sort_values("Date").reset_index(drop=True)

# Merge asset and macro features
combined_features = asset_features.merge(macro_features, on="Date", how="left", validate="many_to_one")
combined_features = combined_features.sort_values(["Date", "Ticker"]).reset_index(drop=True)

# Check missing macro values (exclude BTC and ETH)
macro_columns = [column for column in macro_features.columns if column != "Date"]
non_crypto_mask = ~combined_features["Ticker"].isin(["BTC-USD", "ETH-USD"])
missing_macro_values = combined_features.loc[non_crypto_mask, macro_columns].isna().sum()
missing_macro_values = missing_macro_values[missing_macro_values > 0]

if not missing_macro_values.empty:
    print("Missing macro values after merge (excluding BTC and ETH):")
    display(missing_macro_values)
else:
    # Remove rows with missing macro features where ticker is crypto
    combined_features = (combined_features.dropna(subset=macro_columns).reset_index(drop=True))
    print("No missing macro values after merge.")

# Check asset-date uniqueness
duplicate_asset_rows = (combined_features.duplicated(subset=["Date", "Ticker"]).sum())
print(f"Duplicate Date-Ticker rows: {duplicate_asset_rows}")

# Save combined features
COMBINED_FEATURES_FILE = (FEATURES_DATA_DIR / "combined_features.csv")
combined_features.to_csv(COMBINED_FEATURES_FILE, index=False)
print(f"Saved combined features to: {COMBINED_FEATURES_FILE}")

No missing macro values after merge.
Duplicate Date-Ticker rows: 0
Saved combined features to: features_data\combined_features.csv
